In [33]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np

In [ ]:
folder=r'Path'
OFolder=r'Output Folder Path'

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.csv')  in f:
                    file_list.append(f)
file_list

In [36]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.csv')  in f:
                    file_list.append(os.path.join(root, f))
file_list

In [ ]:
Brands=[]
for name in file_list:
    Brands.append(name.split("\\")[-1].split("_")[0])

Brands=list(set(Brands))
Brands

In [ ]:
location=r"filepath"

In [ ]:
Brands

In [ ]:

for Brand in Brands:
    try:
        #print(location+"\\"+Brand+"\\"+Brand+"_PartCat_Apps.csv")
        #print("Executing for "+Brand)
        df_PartCat=pd.read_csv(location+"\\"+Brand+"\\"+Brand+"_PartCat_Apps.csv").drop_duplicates()
        df_JNP=pd.read_csv(location+"\\"+Brand+"\\"+Brand+"_JNP_Apps.csv").drop_duplicates()
        # del df_PartCat['MfrLabel']
        # del df_JNP['MfrLabel']
        df_merged=df_JNP.merge(df_PartCat, how="outer", on=['Key'], indicator=True ,suffixes=("_JNP", "_PartCat"))
        d={"left_only":"Present Only in JNP", "right_only":"Present only in PartCat","both":"Present in Both Databases"}
        df_merged['_merge'] = df_merged['_merge'].map(d)
        df_merged.to_csv(location+"\\"+Brand+"\\"+Brand+"_Comparison_Output.csv", index=False)
        print("Completed for "+Brand)
    except:
        continue
        

In [ ]:
df_merged

In [ ]:
for file in file_list:
    for brand in Brands:
        if "JNP" in file and brand in file:
            print(file)
            df_JNP=pd.read_csv(file)
        if "PartCat" in file and brand in file:
            print(file+"!")
            df_PartCat=pd.read_csv(file)
        df_output=df_JNP.merge(df_PartCat["Key"], how="outer", on=['Key'], indicator=True)

In [ ]:
df_output

In [ ]:
df_output

In [ ]:
file_link[1]

In [ ]:
df_in=pd.read_excel(file_link[1],sheet_name="Export")
df_in

In [9]:
df_cleaned=pd.melt(df_in,id_vars=['PRODUCT_NAME', 'ITEM_TYPE'], value_vars=[ 'Supplier Item Number Coverage',
       'Supplier ID Coverage', 'Item Type Coverage',
       'Expanded Item Number Coverage', 'Item Number Compressed Coverage',
       'Online Display Name Coverage', 
       'Interchange Coverage', 'OE Interchange Coverage',
       'Shipping Exclusions? Coverage', 'Salability Restrictions Coverage',
       'Shipping Exclusions info Coverage', 'Harmonize Code Coverage',
       'Proper Shipping Name Coverage', 'Hazardous Class or Division Coverage',
       'UN ID Number Coverage', 'Package Group Coverage',
       'Label Codes Coverage', 'Application Item? Coverage',
       'UNSPSC Code Coverage', 'HQ Line Coverage', 'Group Code Coverage',
       'Jobber F.O.Q. Coverage', 'Jobber Std Pkg Coverage',
       'Product Classification Coverage', 'DC Cost (USD) Coverage',
       'Goldenrod Price Coverage', 'Blue Price Coverage',
       'Master Installer Coverage', 'Green Price Coverage',
       'Red/Retail (USD) Coverage', 'List (USD) Coverage',
       'Core Pricing? Coverage', 'Per Car Qty Coverage',
       'Supplier UOM Coverage', 'Disposition Code Coverage',
       'Date Product Available Coverage', 'ORM-D Coverage',
       'Regulated? Coverage', 'Standard Package Coverage',
       'Unit Desg Coverage', 'ws_UPS Coverage',
       'Is Sold To Consumer-US Coverage',
       'Quantity of Eaches in Package Coverage', 'DC Core (USD) Coverage',
       'List Core (USD) Coverage', 'Jobber Core Coverage',
       'Core Code Coverage', 'Primary Image Coverage',
       'Prop65 Statement Coverage', 'Prop 65 PDF Coverage', 'SDS PDF Coverage',
       'Warranty PDF Coverage', 'Product Features Coverage',
       'Features and Benefits 1 Coverage', 'Features and Benefits 2 Coverage',
       'Features and Benefits 3 Coverage', 'Brand Coverage',
       'Prop65 Required Flag Coverage', 'Is UPC Required? Coverage',
       'Single UPC Coverage', 'Reason for No UPC Coverage', 'Height Coverage',
       'Width Coverage', 'Length/Depth Coverage', 'Weight Coverage'],value_name='input_value', ignore_index=False)

In [ ]:
df_cleaned

In [11]:
df_cleaned['Key']="BPI"+df_cleaned['ITEM_TYPE']+df_cleaned['PRODUCT_NAME']+df_cleaned['variable']

In [12]:
df_cleaned=df_cleaned[['PRODUCT_NAME','ITEM_TYPE','variable','Key','input_value']]

In [ ]:
df_db1=pd.read_excel(r'path/filename',sheet_name="Part_1",skiprows=[0],header=[1])
df_db1

In [ ]:
df_db2=pd.read_excel(r'path/filename',sheet_name="Part_2",skiprows=[0],header=[1])
df_db2

In [32]:
df_db=pd.concat([df_db1, df_db2])

In [33]:
df_db=df_db[df_db['Key'].str.startswith('BPI')]

In [34]:
df_db=df_db[['Key','Date']]
df_db['db_value']=np.where(df_db['Date'].isna(),0,1 )

In [35]:
df_db=df_db[['Key','db_value']]

In [ ]:
df_db['db_value'].value_counts()

In [37]:
df=pd.merge(df_db,df_cleaned,how='inner')

In [38]:
df['Difference']=np.where(df['db_value']!=df['input_value'],1,0)

In [ ]:
df_final=df[df['Difference']==1]
df_final

In [ ]:
df_final.to_excel(r'path/filename.xlsx',sheet_name='Comparision')